
# 7. Summary

Этот notebook собирает **только табличный summary** по уже готовым артефактам эксперимента.

Он не перезапускает:
- `2-infer.ipynb`
- `3-project.ipynb`
- `4-align.ipynb`
- `5-score.ipynb`

Назначение notebook:
- дать компактный, но исчерпывающий appendix для статьи;
- зафиксировать входные метрики датасета;
- зафиксировать выходные метрики по inference / projection / alignment / scoring;
- отдельно показать alignment-centric результаты, на которых видно эффект `feedback`.


In [2]:
from __future__ import annotations

import json
import re
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 220)
pd.set_option('display.max_colwidth', 120)

SCENARIO_ORDER = ['control', 'guided', 'feedback']
MODEL_SHORT_NAMES = {
    'google/gemini-2.5-flash-lite': 'Gemini',
    'meta-llama/llama-3.2-3b-instruct': 'Llama',
    'mistralai/ministral-3b-2512': 'Ministral',
}
BOOLEAN_PREFIXES = (
    'is',
    'has',
    'can',
    'should',
    'supports',
    'allow',
    'allows',
    'enabled',
    'enable',
    'uses',
    'contains',
    'includes',
    'requires',
    'was',
    'were',
    'will',
)
GENERIC_NAMES = {
    'data',
    'info',
    'details',
    'item',
    'items',
    'object',
    'objects',
    'value',
    'values',
    'entry',
    'entries',
    'record',
    'records',
    'result',
    'results',
}
SEMANTIC_MATCHERS = [('valentine', 'coma_py'), ('bdikit', 'coma')]


def find_repo_root(start: Path | None = None) -> Path:
    cursor = (start or Path.cwd()).resolve()
    for candidate in [cursor, *cursor.parents]:
        if (candidate / 'artifacts' / 'score' / 'pair-scores.csv').exists():
            return candidate
    raise FileNotFoundError(
        'Could not find repo root containing artifacts/score/pair-scores.csv'
    )


def short_model_name(model: str) -> str:
    return MODEL_SHORT_NAMES.get(model, model.rsplit('/', 1)[-1])


def tokenize_identifier(name: str) -> list[str]:
    return [
        token.lower()
        for token in re.findall(r'[A-Z]?[a-z]+|[A-Z]+(?![a-z])|\d+', str(name))
        if token
    ]


repo_root = find_repo_root()
source_docs_dir = repo_root / 'data' / 'soruce' / 'documents'
source_matrix = pd.read_json(
    repo_root / 'artifacts' / 'sources' / 'source-matrix.json'
)
infer_results = pd.DataFrame(
    json.loads(
        (
            repo_root / 'artifacts' / 'infer' / 'infer-results.json'
        ).read_text()
    )
)
model_index = pd.read_csv(
    repo_root / 'artifacts' / 'project' / 'extracted' / 'model_index.csv'
)
elements = pd.read_csv(
    repo_root / 'artifacts' / 'project' / 'extracted' / 'elements.csv'
)
pairs = pd.read_csv(repo_root / 'artifacts' / 'align' / 'pairs.csv')
alignment_candidates = pd.read_csv(
    repo_root / 'artifacts' / 'align' / 'alignment-candidates.csv'
)
pair_scores = pd.read_csv(repo_root / 'artifacts' / 'score' / 'pair-scores.csv')
model_scores = pd.read_csv(repo_root / 'artifacts' / 'score' / 'model-scores.csv')
scenario_scores = pd.read_csv(
    repo_root / 'artifacts' / 'score' / 'scenario-scores.csv'
)
scenario_matrix_scores = pd.read_csv(
    repo_root / 'artifacts' / 'score' / 'scenario-matrix-scores.csv'
)

same_scenario_scores = pair_scores[
    pair_scores['source_condition'] == pair_scores['target_condition']
].copy()
semantic_same_scenario_scores = same_scenario_scores[
    [
        (backend, method) in SEMANTIC_MATCHERS
        for backend, method in zip(
            same_scenario_scores['backend'],
            same_scenario_scores['method'],
            strict=True,
        )
    ]
].copy()
valentine_coma_scores = same_scenario_scores[
    (same_scenario_scores['backend'] == 'valentine')
    & (same_scenario_scores['method'] == 'coma_py')
].copy()

source_docs_rows = []
for path in sorted(source_docs_dir.glob('*.md')):
    if path.name.lower() == 'readme.md':
        continue
    text = path.read_text(encoding='utf-8')
    source_docs_rows.append(
        {
            'document': path.name,
            'document_stem': path.stem,
            'line_count': len(text.splitlines()),
            'word_count': len(text.split()),
            'char_count': len(text),
            'byte_count': len(text.encode('utf-8')),
        }
    )
source_docs = pd.DataFrame(source_docs_rows).sort_values(
    'document_stem',
    ignore_index=True,
)

artifact_rows = []
for _, row in model_index.iterrows():
    code_path = repo_root / row['final_cs_path']
    text = code_path.read_text(encoding='utf-8')
    parse_error_count = (
        int(row['parse_error_count'])
        if not pd.isna(row['parse_error_count'])
        else 0
    )
    artifact_rows.append(
        {
            'model_id': row['model_id'],
            'source_document_id': row['source_document_id'],
            'condition': row['condition'],
            'producer_id': row['producer_id'],
            'producer_short': short_model_name(row['producer_id']),
            'line_count': len(text.splitlines()),
            'char_count': len(text),
            'byte_count': len(text.encode('utf-8')),
            'parse_error_count': parse_error_count,
        }
    )
artifact_stats = pd.DataFrame(artifact_rows)

artifact_element_counts = (
    elements.groupby(['model_id', 'element_kind'])['element_id']
    .count()
    .unstack(fill_value=0)
    .reset_index()
)
model_index_keys = model_index[
    ['model_id', 'source_document_id', 'condition', 'producer_id']
].drop_duplicates()
artifact_summary = (
    artifact_stats.merge(
        model_index_keys,
        on=['model_id', 'source_document_id', 'condition', 'producer_id'],
        how='left',
    )
    .merge(artifact_element_counts, on='model_id', how='left')
    .fillna({'property': 0, 'type': 0, 'relation': 0, 'enum_member': 0})
)
artifact_summary['total_elements'] = (
    artifact_summary['property']
    + artifact_summary['type']
    + artifact_summary['relation']
    + artifact_summary['enum_member']
)
artifact_summary = artifact_summary.sort_values(
    ['source_document_id', 'condition', 'producer_id'],
    ignore_index=True,
)

metric_catalog = pd.DataFrame(
    [
        {
            'metric': 'source_coverage',
            'definition': (
                'Share of source elements that participate in at least one '
                'alignment candidate match.'
            ),
        },
        {
            'metric': 'target_coverage',
            'definition': (
                'Share of target elements that participate in at least one '
                'alignment candidate match.'
            ),
        },
        {
            'metric': 'coverage_f1',
            'definition': 'Harmonic mean of source_coverage and target_coverage.',
        },
        {
            'metric': 'top1_mean_score',
            'definition': (
                'Mean score of the best-ranked candidate match per source '
                'element.'
            ),
        },
        {
            'metric': 'pair_alignment_f1',
            'definition': 'coverage_f1 multiplied by top1_mean_score.',
        },
    ]
)

display(Markdown(f'**Repo root:** `{repo_root}`'))
display(metric_catalog)

**Repo root:** `/home/bearpro/source/personal/llm-in-lsp-experiment`

,metric,definition
0,source_coverage,Share of source elements that participate in at least one alignment candidate match.
1,target_coverage,Share of target elements that participate in at least one alignment candidate match.
2,coverage_f1,Harmonic mean of source_coverage and target_coverage.
3,top1_mean_score,Mean score of the best-ranked candidate match per source element.
4,pair_alignment_f1,coverage_f1 multiplied by top1_mean_score.



## Experiment design and input dataset

Этот раздел фиксирует то, что в академической статье обычно оказывается в `Experimental Setup`:
- число документов;
- число сценариев инференса;
- число моделей;
- полную экспериментальную матрицу;
- размеры исходного корпуса.


In [3]:
scenario_catalog = pd.DataFrame(
    [
        {
            'scenario': 'control',
            'description': (
                'Direct generation of a C# domain model from the source '
                'document without the extra feedback stage.'
            ),
        },
        {
            'scenario': 'guided',
            'description': 'Generation with the structured guided prompt templates.',
        },
        {
            'scenario': 'feedback',
            'description': (
                'Guided generation -> standalone feedback by GPT-5.4-mini '
                'over the C# file only -> revision by the original '
                'generator model.'
            ),
        },
    ]
)

model_catalog = pd.DataFrame({'model': sorted(source_matrix['model'].unique())})
model_catalog = model_catalog.assign(
    model_short=lambda frame: frame['model'].map(short_model_name)
)

design_summary = pd.DataFrame(
    [
        {
            'document_count': int(source_docs['document_stem'].nunique()),
            'scenario_count': int(source_matrix['scenario'].nunique()),
            'model_count': int(source_matrix['model'].nunique()),
            'runs_per_combination': int(model_index['run'].max()),
            'expected_inference_runs': int(len(source_matrix)),
            'completed_inference_runs': int(len(infer_results)),
        }
    ]
)

dataset_totals = pd.DataFrame(
    [
        {
            'total_source_lines': int(source_docs['line_count'].sum()),
            'total_source_words': int(source_docs['word_count'].sum()),
            'total_source_chars': int(source_docs['char_count'].sum()),
            'total_source_bytes': int(source_docs['byte_count'].sum()),
        }
    ]
)

source_matrix_counts = (
    source_matrix.groupby(['scenario', 'model'])['document_stem']
    .count()
    .unstack(fill_value=0)
    .reindex(index=SCENARIO_ORDER)
)

display(Markdown('### Design summary'))
display(design_summary)
display(Markdown('### Scenario catalog'))
display(scenario_catalog)
display(Markdown('### Model catalog'))
display(model_catalog)
display(Markdown('### Source corpus totals'))
display(dataset_totals)
display(Markdown('### Source documents'))
display(source_docs)
display(Markdown('### Source matrix counts'))
display(source_matrix_counts)

### Design summary

,document_count,scenario_count,model_count,runs_per_combination,expected_inference_runs,completed_inference_runs
0,6,3,3,1,54,54


### Scenario catalog

,scenario,description
0,control,Direct generation of a C# domain model from the source document without the extra feedback stage.
1,guided,Generation with the structured guided prompt templates.
2,feedback,Guided generation -> standalone feedback by GPT-5.4-mini over the C# file only -> revision by the original generator...


### Model catalog

...

,total_source_lines,total_source_words,total_source_chars,total_source_bytes
0,189,3238,20590,20602


### Source documents

,document,document_stem,line_count,word_count,char_count,byte_count
0,framework-laptop-13-specs.md,framework-laptop-13-specs,21,480,2752,2754
1,github-create-issue-api.md,github-create-issue-api,28,453,3037,3039
2,ikea-billy-mainual.md,ikea-billy-mainual,45,613,3710,3712
3,king-arthur-pancakes.md,king-arthur-pancakes,30,412,2419,2421
4,standard-fragment.md,standard-fragment,23,700,4123,4125
5,stripe-payment-intent.md,stripe-payment-intent,42,580,4549,4551


### Source matrix counts

model,google/gemini-2.5-flash-lite,meta-llama/llama-3.2-3b-instruct,mistralai/ministral-3b-2512
scenario,,,
control,6,6,6
guided,6,6,6
feedback,6,6,6



## Inference outputs

Здесь фиксируются итоговые характеристики уже сгенерированных `final.cs` файлов:
- полнота выполнения матрицы;
- размеры исходников;
- синтаксические ошибки Roslyn-парсинга;
- полная artifact-level таблица по всем 54 артефактам.


In [4]:
infer_status_table = (
    infer_results['status']
    .value_counts()
    .rename_axis('status')
    .reset_index(name='count')
)

output_size_by_scenario = (
    artifact_summary.groupby('condition')[
        ['line_count', 'char_count', 'byte_count', 'parse_error_count', 'total_elements']
    ]
    .agg(['mean', 'median', 'min', 'max'])
    .round(2)
)

parse_error_by_scenario = (
    artifact_summary.groupby('condition')['parse_error_count']
    .agg(['sum', 'mean', 'median', 'max'])
    .round(3)
)
parse_error_by_scenario['zero_error_rate'] = (
    artifact_summary.groupby('condition')['parse_error_count']
    .apply(lambda series: float((series == 0).mean()))
    .round(3)
)

parse_error_by_model = (
    artifact_summary.groupby('producer_id')['parse_error_count']
    .agg(['sum', 'mean', 'median', 'max'])
    .round(3)
)
parse_error_by_model['zero_error_rate'] = (
    artifact_summary.groupby('producer_id')['parse_error_count']
    .apply(lambda series: float((series == 0).mean()))
    .round(3)
)

artifact_table = artifact_summary[
    [
        'source_document_id',
        'condition',
        'producer_short',
        'line_count',
        'char_count',
        'byte_count',
        'parse_error_count',
        'property',
        'type',
        'relation',
        'enum_member',
        'total_elements',
    ]
].rename(columns={'producer_short': 'model'})

display(Markdown('### Inference status'))
display(infer_status_table)
display(Markdown('### Output size statistics by scenario'))
display(output_size_by_scenario)
display(Markdown('### Parse errors by scenario'))
display(parse_error_by_scenario)
display(Markdown('### Parse errors by model'))
display(parse_error_by_model)
display(Markdown('### Artifact-level output table (all 54 generated models)'))
display(artifact_table)

### Inference status

,status,count
0,completed,54


### Output size statistics by scenario

line_count                 char_count                     byte_count                     parse_error_count                total_elements                
                mean median min  max       mean  median  min    max       mean  median  min    max              mean median min max           mean median min  max
condition                                                                                                                                                         
control       132.89  114.5  49  359    4077.56  3202.5  898   9639    4077.72  3202.5  898   9639              0.00    0.0   0   0          58.72   58.0  30  104
feedback       98.28   75.5  33  346    6003.22  1895.5  791  62966    6008.06  1895.5  791  62966              2.28    0.0   0  17          54.00   46.0  26   97
guided         89.17   68.0  24  360    2253.22  1653.0  532   8707    2253.22  1653.0  532   8707              0.22    0.0   0   2          53.11   47.5  20  104

### Parse errors by scenario

,sum,mean,median,max,zero_error_rate
condition,,,,,
control,0,0.000,0.0,0,1.000
feedback,41,2.278,0.0,17,0.833
guided,4,0.222,0.0,2,0.889


### Parse errors by model

,sum,mean,median,max,zero_error_rate
producer_id,,,,,
google/gemini-2.5-flash-lite,0,0.000,0.0,0,1.000
meta-llama/llama-3.2-3b-instruct,43,2.389,0.0,17,0.778
mistralai/ministral-3b-2512,2,0.111,0.0,2,0.944


### Artifact-level output table (all 54 generated models)

,source_document_id,condition,model,line_count,char_count,byte_count,parse_error_count,property,type,relation,enum_member,total_elements
0,framework-laptop-13-specs,control,Gemini,118,3337,3337,0,52,14,11,6,83
1,framework-laptop-13-specs,control,Llama,66,1794,1794,0,32,6,2,8,48
2,framework-laptop-13-specs,control,Ministral,210,5497,5497,0,49,12,8,8,77
3,framework-laptop-13-specs,feedback,Gemini,110,2892,2892,0,48,13,9,9,79
4,framework-laptop-13-specs,feedback,Llama,40,1861,1861,0,34,1,0,0,35
5,framework-laptop-13-specs,feedback,Ministral,142,3920,3920,0,58,16,14,9,97
6,framework-laptop-13-specs,guided,Gemini,104,2774,2774,0,51,13,13,0,77
7,framework-laptop-13-specs,guided,Llama,40,1819,1819,0,34,1,0,0,35
8,framework-laptop-13-specs,guided,Ministral,111,2463,2463,2,58,23,20,3,104
9,github-create-issue-api,control,Gemini,359,9639,9639,0,60,8,6,4,78



## Projection and alignment workload

Этот раздел нужен для `Implementation Details` / `Results Volume`:
- сколько элементов получилось после проекции в табличную модель;
- сколько directed model pairs было построено;
- сколько alignment candidates реально посчитано;
- как workload распределён по методам и projection layers.


In [5]:
element_counts_by_scenario = (
    elements.groupby('condition')['element_id']
    .count()
    .rename('element_count')
    .to_frame()
)

element_kinds_by_scenario = (
    elements.groupby(['condition', 'element_kind'])['element_id']
    .count()
    .unstack(fill_value=0)
)

element_kinds_overall = (
    elements['element_kind']
    .value_counts()
    .rename_axis('element_kind')
    .reset_index(name='count')
)

pair_matrix = (
    pairs.groupby(['source_condition', 'target_condition'])['pair_id']
    .count()
    .unstack(fill_value=0)
    .reindex(index=SCENARIO_ORDER, columns=SCENARIO_ORDER)
)

candidate_counts_by_method = (
    alignment_candidates.groupby(['backend', 'method'])['pair_id']
    .count()
    .rename('candidate_count')
    .sort_values(ascending=False)
    .reset_index()
)

candidate_counts_by_projection = (
    alignment_candidates.groupby(['projection_layer', 'projection_mode'])['pair_id']
    .count()
    .rename('candidate_count')
    .sort_values(ascending=False)
    .reset_index()
)

workload_fact_sheet = pd.DataFrame(
    [
        {
            'projected_models': int(model_index['model_id'].nunique()),
            'projected_elements': int(len(elements)),
            'directed_alignment_pairs': int(len(pairs)),
            'alignment_candidates': int(len(alignment_candidates)),
            'scored_pair_slices': int(len(pair_scores)),
            'scored_model_slices': int(len(model_scores)),
            'scored_scenario_slices': int(len(scenario_scores)),
            'scored_scenario_matrix_slices': int(len(scenario_matrix_scores)),
        }
    ]
)

display(Markdown('### Workload fact sheet'))
display(workload_fact_sheet)
display(Markdown('### Element counts by scenario'))
display(element_counts_by_scenario)
display(Markdown('### Element kinds by scenario'))
display(element_kinds_by_scenario)
display(Markdown('### Element kinds overall'))
display(element_kinds_overall)
display(Markdown('### Directed pair matrix'))
display(pair_matrix)
display(Markdown('### Alignment candidates by method'))
display(candidate_counts_by_method)
display(Markdown('### Alignment candidates by projection'))
display(candidate_counts_by_projection)

### Workload fact sheet

,projected_models,projected_elements,directed_alignment_pairs,alignment_candidates,scored_pair_slices,scored_model_slices,scored_scenario_slices,scored_scenario_matrix_slices
0,54,2985,432,298745,13165,1857,670,1929


### Element counts by scenario

,element_count
condition,
control,1057
feedback,972
guided,956


### Element kinds by scenario

...

target_condition,control,guided,feedback
source_condition,,,
control,36,54,54
guided,54,36,54
feedback,54,54,36


### Alignment candidates by method

,backend,method,candidate_count
0,magneto,native_zero_download,101055
1,bdikit,jaccard_distance,36275
2,bdikit,similarity_flooding,36275
3,bdikit,coma,33552
4,valentine,similarity_flooding,26988
5,bdikit,distribution_based,23483
6,valentine,coma_py,17469
7,valentine,jaccard_distance,15788
8,bdikit,cupid,3327
9,valentine,distribution_based,3225


### Alignment candidates by projection

,projection_layer,projection_mode,candidate_count
0,property,path_plus_metadata_values,130284
1,property,path_only,127851
2,type,members_as_values,25904
3,relation,path_plus_metadata_values,14706



## Alignment outcome metrics

Это главный раздел с метриками качества выравнивания.

Сначала идут общие табличные сводки по всем методам, затем более целевой alignment-centric срез:
- same-scenario overall;
- full scenario matrix;
- same-scenario breakdown by method, model, and document.


In [6]:
same_scenario_overall = (
    same_scenario_scores.groupby('source_condition')[
        [
            'source_coverage',
            'target_coverage',
            'coverage_f1',
            'top1_mean_score',
            'pair_alignment_f1',
        ]
    ]
    .mean()
    .reindex(SCENARIO_ORDER)
    .round(4)
)

scenario_matrix_overall = (
    scenario_matrix_scores.groupby(['source_condition', 'target_condition'])[
        ['mean_coverage_f1', 'mean_top1_score', 'mean_pair_alignment_f1']
    ]
    .mean()
    .round(4)
)

same_scenario_by_model = (
    same_scenario_scores.groupby(['source_producer_id', 'source_condition'])[
        ['coverage_f1', 'top1_mean_score', 'pair_alignment_f1']
    ]
    .mean()
    .reset_index()
)

same_scenario_model_pair_alignment = (
    same_scenario_by_model.pivot_table(
        index='source_producer_id',
        columns='source_condition',
        values='pair_alignment_f1',
    )
    .reindex(columns=SCENARIO_ORDER)
    .round(4)
)

same_scenario_by_document = (
    same_scenario_scores.groupby(['source_document_id', 'source_condition'])[
        ['coverage_f1', 'top1_mean_score', 'pair_alignment_f1']
    ]
    .mean()
    .reset_index()
)

same_scenario_document_pair_alignment = (
    same_scenario_by_document.pivot_table(
        index='source_document_id',
        columns='source_condition',
        values='pair_alignment_f1',
    )
    .reindex(columns=SCENARIO_ORDER)
    .round(4)
)

method_pair_alignment = (
    same_scenario_scores.groupby(['backend', 'method', 'source_condition'])[
        'pair_alignment_f1'
    ]
    .mean()
    .reset_index()
)
method_pair_alignment_table = (
    method_pair_alignment.pivot_table(
        index=['backend', 'method'],
        columns='source_condition',
        values='pair_alignment_f1',
    )
    .reindex(columns=SCENARIO_ORDER)
)
method_pair_alignment_table['guided_minus_control'] = (
    method_pair_alignment_table['guided'] - method_pair_alignment_table['control']
)
method_pair_alignment_table['feedback_minus_guided'] = (
    method_pair_alignment_table['feedback'] - method_pair_alignment_table['guided']
)
method_pair_alignment_table['feedback_minus_control'] = (
    method_pair_alignment_table['feedback'] - method_pair_alignment_table['control']
)
method_pair_alignment_table = method_pair_alignment_table.sort_values(
    'feedback_minus_guided',
    ascending=False,
).round(4)

display(Markdown('### Same-scenario overall (all methods)'))
display(same_scenario_overall)
display(Markdown('### Full scenario matrix overall'))
display(scenario_matrix_overall)
display(Markdown('### Same-scenario Pair alignment F1 by source model (all methods)'))
display(same_scenario_model_pair_alignment)
display(Markdown('### Same-scenario Pair alignment F1 by source document (all methods)'))
display(same_scenario_document_pair_alignment)
display(Markdown('### Same-scenario Pair alignment F1 by alignment method'))
display(method_pair_alignment_table)

### Same-scenario overall (all methods)

,source_coverage,target_coverage,coverage_f1,top1_mean_score,pair_alignment_f1
source_condition,,,,,
control,0.7468,0.5128,0.5336,0.5646,0.2853
guided,0.7621,0.5434,0.5410,0.5387,0.2821
feedback,0.7753,0.5506,0.5557,0.5175,0.2811


### Full scenario matrix overall

mean_coverage_f1  mean_top1_score  mean_pair_alignment_f1
source_condition target_condition                                                           
control          control                     0.5339           0.5768                  0.2940
                 feedback                    0.4999           0.4930                  0.2342
                 guided                      0.5169           0.5062                  0.2536
feedback         control                     0.4930           0.4941                  0.2281
                 feedback                    0.5571           0.5391                  0.2916
                 guided                      0.5983           0.5976                  0.3536
guided           control                     0.5074           0.5055                  0.2438
                 feedback                    0.5978           0.5958                  0.3512
                 guided                      0.5340           0.5575                  0.2848

### Same-scenario Pair alignment F1 by source model (all methods)

source_condition,control,guided,feedback
source_producer_id,,,
google/gemini-2.5-flash-lite,0.2912,0.2942,0.2993
meta-llama/llama-3.2-3b-instruct,0.2624,0.2283,0.2346
mistralai/ministral-3b-2512,0.3027,0.3198,0.3055


### Same-scenario Pair alignment F1 by source document (all methods)

source_condition,control,guided,feedback
source_document_id,,,
framework-laptop-13-specs,0.2666,0.1825,0.1717
github-create-issue-api,0.3672,0.3493,0.2901
ikea-billy-mainual,0.1530,0.2670,0.3275
king-arthur-pancakes,0.3576,0.1725,0.1473
standard-fragment,0.2357,0.2239,0.2545
stripe-payment-intent,0.3079,0.4525,0.4344


### Same-scenario Pair alignment F1 by alignment method

source_condition                control  guided  feedback  guided_minus_control  feedback_minus_guided  feedback_minus_control
backend   method                                                                                                              
valentine distribution_based     0.5162  0.4463    0.5341               -0.0699                 0.0878                  0.0179
          coma_py                0.3595  0.3983    0.4247                0.0388                 0.0264                  0.0653
bdikit    coma                   0.4749  0.5319    0.5520                0.0570                 0.0201                  0.0771
          cupid                  0.5469  0.5238    0.5426               -0.0231                 0.0189                 -0.0042
valentine similarity_flooding    0.0767  0.0596    0.0660               -0.0170                 0.0064                 -0.0106
bdikit    similarity_flooding    0.0958  0.0818    0.0880               -0.0140                 0.0061                 -0.0079
magneto   native_zero_download   0.0000  0.0000    0.0000                0.0000                 0.0000                  0.0000
valentine cupid                  0.4018  0.4243    0.4004                0.0225                -0.0239                 -0.0014
bdikit    distribution_based     0.2943  0.2978    0.2688                0.0035                -0.0290                 -0.0255
valentine jaccard_distance       0.3995  0.3912    0.3540               -0.0083                -0.0372                 -0.0455
bdikit    jaccard_distance       0.3564  0.3316    0.2931               -0.0248                -0.0385                 -0.0633


## Alignment-centric slice for the feedback hypothesis

Глобальное среднее по всем методам смешивает structural/semantic matcher'ы и более lexical matcher'ы.

Чтобы проверить именно гипотезу `feedback improves model alignment`, ниже даются два более целевых среза:
- **semantic matcher slice**: `valentine:coma_py` + `bdikit:coma`;
- **primary table**: только `valentine:coma_py`, агрегированный по всем документам и projection layers, но разложенный по directed model pairs.


In [7]:
semantic_slice_overall = (
    semantic_same_scenario_scores.groupby('source_condition')[
        ['coverage_f1', 'top1_mean_score', 'pair_alignment_f1']
    ]
    .mean()
    .reindex(SCENARIO_ORDER)
    .round(4)
)

semantic_slice_by_model = (
    semantic_same_scenario_scores.groupby(
        ['source_producer_id', 'source_condition']
    )['pair_alignment_f1']
    .mean()
    .reset_index()
    .pivot_table(
        index='source_producer_id',
        columns='source_condition',
        values='pair_alignment_f1',
    )
    .reindex(columns=SCENARIO_ORDER)
    .round(4)
)

valentine_coma_pair_table = (
    valentine_coma_scores.groupby(
        ['source_producer_id', 'target_producer_id', 'source_condition']
    )['pair_alignment_f1']
    .mean()
    .reset_index()
    .pivot_table(
        index=['source_producer_id', 'target_producer_id'],
        columns='source_condition',
        values='pair_alignment_f1',
    )
    .reindex(columns=SCENARIO_ORDER)
    .round(4)
)

valentine_coma_doc_table = (
    valentine_coma_scores.groupby(
        ['source_document_id', 'source_condition']
    )[['coverage_f1', 'pair_alignment_f1']]
    .mean()
    .reset_index()
)
valentine_coma_doc_pair_alignment = (
    valentine_coma_doc_table.pivot_table(
        index='source_document_id',
        columns='source_condition',
        values='pair_alignment_f1',
    )
    .reindex(columns=SCENARIO_ORDER)
    .round(4)
)

feedback_gt_guided = (
    valentine_coma_pair_table['feedback'] > valentine_coma_pair_table['guided']
).all()
guided_gt_control = (
    valentine_coma_pair_table['guided'] > valentine_coma_pair_table['control']
).all()
valentine_coma_fact_sheet = pd.DataFrame(
    [
        {
            'directed_model_pair_count': int(len(valentine_coma_pair_table)),
            'all_feedback_gt_guided': bool(feedback_gt_guided),
            'all_guided_gt_control': bool(guided_gt_control),
        }
    ]
)

display(Markdown('### Semantic matcher slice overall'))
display(semantic_slice_overall)
display(Markdown('### Semantic matcher slice: Pair alignment F1 by source model'))
display(semantic_slice_by_model)
display(Markdown('### Primary alignment table: `valentine:coma_py` by directed model pair'))
display(valentine_coma_pair_table)
display(Markdown('### Primary alignment fact sheet'))
display(valentine_coma_fact_sheet)
display(Markdown('### `valentine:coma_py` by source document'))
display(valentine_coma_doc_pair_alignment)

### Semantic matcher slice overall

,coverage_f1,top1_mean_score,pair_alignment_f1
source_condition,,,
control,0.5751,0.7159,0.4172
guided,0.6268,0.7258,0.4651
feedback,0.6479,0.7373,0.4886


### Semantic matcher slice: Pair alignment F1 by source model

source_condition,control,guided,feedback
source_producer_id,,,
google/gemini-2.5-flash-lite,0.4266,0.4873,0.5158
meta-llama/llama-3.2-3b-instruct,0.3874,0.3708,0.4123
mistralai/ministral-3b-2512,0.4398,0.5306,0.5287


### Primary alignment table: `valentine:coma_py` by directed model pair

source_condition                                                   control  guided  feedback
source_producer_id               target_producer_id                                         
google/gemini-2.5-flash-lite     meta-llama/llama-3.2-3b-instruct   0.3182  0.3557    0.3684
                                 mistralai/ministral-3b-2512        0.4351  0.4808    0.4899
meta-llama/llama-3.2-3b-instruct google/gemini-2.5-flash-lite       0.3182  0.3557    0.3684
                                 mistralai/ministral-3b-2512        0.3409  0.3459    0.3992
mistralai/ministral-3b-2512      google/gemini-2.5-flash-lite       0.4351  0.4808    0.4903
                                 meta-llama/llama-3.2-3b-instruct   0.3244  0.3459    0.3992

### Primary alignment fact sheet

,directed_model_pair_count,all_feedback_gt_guided,all_guided_gt_control
0,6,True,True


### `valentine:coma_py` by source document

source_condition,control,guided,feedback
source_document_id,,,
framework-laptop-13-specs,0.4003,0.2629,0.2712
github-create-issue-api,0.5795,0.4728,0.4462
ikea-billy-mainual,0.1525,0.4011,0.4693
king-arthur-pancakes,0.4027,0.2455,0.2424
standard-fragment,0.2157,0.3212,0.3552
stripe-payment-intent,0.4279,0.6631,0.6632


In [8]:
from random import Random


def _mean(values):
    return sum(values) / len(values)


def _quantile(sorted_values, q):
    idx = (len(sorted_values) - 1) * q
    lo = int(idx)
    hi = min(lo + 1, len(sorted_values) - 1)
    frac = idx - lo
    return sorted_values[lo] * (1 - frac) + sorted_values[hi] * frac


def _bootstrap_unpaired(control_values, feedback_values, *, rounds=20_000, seed=7):
    rng = Random(seed)
    n_control = len(control_values)
    n_feedback = len(feedback_values)
    abs_deltas = []
    rel_deltas = []

    for _ in range(rounds):
        sample_control = [control_values[rng.randrange(n_control)] for _ in range(n_control)]
        sample_feedback = [feedback_values[rng.randrange(n_feedback)] for _ in range(n_feedback)]
        mean_control = _mean(sample_control)
        mean_feedback = _mean(sample_feedback)
        abs_delta = mean_feedback - mean_control
        abs_deltas.append(abs_delta)
        rel_deltas.append(abs_delta / mean_control)

    abs_deltas.sort()
    rel_deltas.sort()
    return {
        'abs_ci_low': _quantile(abs_deltas, 0.025),
        'abs_ci_high': _quantile(abs_deltas, 0.975),
        'rel_ci_low': _quantile(rel_deltas, 0.025),
        'rel_ci_high': _quantile(rel_deltas, 0.975),
    }


def _bootstrap_paired(control_values, feedback_values, *, rounds=20_000, seed=7):
    rng = Random(seed)
    pairs = list(zip(control_values, feedback_values))
    n_pairs = len(pairs)
    abs_deltas = []
    rel_deltas = []

    for _ in range(rounds):
        sample_pairs = [pairs[rng.randrange(n_pairs)] for _ in range(n_pairs)]
        sample_control = [control for control, _ in sample_pairs]
        sample_feedback = [feedback for _, feedback in sample_pairs]
        mean_control = _mean(sample_control)
        mean_feedback = _mean(sample_feedback)
        abs_delta = mean_feedback - mean_control
        abs_deltas.append(abs_delta)
        rel_deltas.append(abs_delta / mean_control)

    abs_deltas.sort()
    rel_deltas.sort()
    return {
        'abs_ci_low': _quantile(abs_deltas, 0.025),
        'abs_ci_high': _quantile(abs_deltas, 0.975),
        'rel_ci_low': _quantile(rel_deltas, 0.025),
        'rel_ci_high': _quantile(rel_deltas, 0.975),
    }


def _permutation_p_value(control_values, feedback_values, *, rounds=20_000, seed=7):
    rng = Random(seed)
    observed = _mean(feedback_values) - _mean(control_values)
    combined = list(control_values) + list(feedback_values)
    n_control = len(control_values)
    exceedances = 0

    for _ in range(rounds):
        shuffled = combined[:]
        rng.shuffle(shuffled)
        delta = _mean(shuffled[n_control:]) - _mean(shuffled[:n_control])
        if abs(delta) >= abs(observed):
            exceedances += 1

    return (exceedances + 1) / (rounds + 1)


def _sign_flip_p_value(deltas, *, rounds=20_000, seed=7):
    rng = Random(seed)
    observed = _mean(deltas)
    exceedances = 0

    for _ in range(rounds):
        signed_mean = _mean([delta if rng.random() < 0.5 else -delta for delta in deltas])
        if abs(signed_mean) >= abs(observed):
            exceedances += 1

    return (exceedances + 1) / (rounds + 1)


semantic_control = semantic_same_scenario_scores.loc[
    semantic_same_scenario_scores['source_condition'] == 'control',
    'pair_alignment_f1',
].tolist()
semantic_feedback = semantic_same_scenario_scores.loc[
    semantic_same_scenario_scores['source_condition'] == 'feedback',
    'pair_alignment_f1',
].tolist()

pair_level_ci = _bootstrap_unpaired(semantic_control, semantic_feedback)
pair_level_mean_control = _mean(semantic_control)
pair_level_mean_feedback = _mean(semantic_feedback)
pair_level_abs_delta = pair_level_mean_feedback - pair_level_mean_control
pair_level_rel_delta = pair_level_abs_delta / pair_level_mean_control

document_level_table = (
    semantic_same_scenario_scores.groupby(['source_document_id', 'source_condition'])['pair_alignment_f1']
    .mean()
    .reset_index()
    .pivot_table(
        index='source_document_id',
        columns='source_condition',
        values='pair_alignment_f1',
    )
    .dropna(subset=['control', 'feedback'])
)
document_control = document_level_table['control'].tolist()
document_feedback = document_level_table['feedback'].tolist()
document_deltas = [
    feedback - control
    for control, feedback in zip(document_control, document_feedback)
]
document_level_ci = _bootstrap_paired(document_control, document_feedback)
document_level_mean_control = _mean(document_control)
document_level_mean_feedback = _mean(document_feedback)
document_level_abs_delta = document_level_mean_feedback - document_level_mean_control
document_level_rel_delta = document_level_abs_delta / document_level_mean_control

semantic_slice_significance = pd.DataFrame(
    [
        {
            'analysis': 'pair-level headline effect',
            'n': len(semantic_control) + len(semantic_feedback),
            'mean_control': pair_level_mean_control,
            'mean_feedback': pair_level_mean_feedback,
            'absolute_delta': pair_level_abs_delta,
            'relative_delta_pct': pair_level_rel_delta * 100,
            'p_value': _permutation_p_value(semantic_control, semantic_feedback),
            'relative_ci_95_pct': f"[{pair_level_ci['rel_ci_low'] * 100:.1f}%; {pair_level_ci['rel_ci_high'] * 100:.1f}%]",
        },
        {
            'analysis': 'document-level sensitivity check',
            'n': len(document_control),
            'mean_control': document_level_mean_control,
            'mean_feedback': document_level_mean_feedback,
            'absolute_delta': document_level_abs_delta,
            'relative_delta_pct': document_level_rel_delta * 100,
            'p_value': _sign_flip_p_value(document_deltas),
            'relative_ci_95_pct': f"[{document_level_ci['rel_ci_low'] * 100:.1f}%; {document_level_ci['rel_ci_high'] * 100:.1f}%]",
        },
    ]
).set_index('analysis')

display(Markdown('### Statistical check for `feedback` vs `control` in the semantic matcher slice'))
display(Markdown('Pair-level analysis reproduces the article headline; document-level analysis is a sensitivity check on the 6 source documents.'))
display(semantic_slice_significance.round({
    'mean_control': 4,
    'mean_feedback': 4,
    'absolute_delta': 4,
    'relative_delta_pct': 2,
    'p_value': 5,
}))


### Statistical check for `feedback` vs `control` in the semantic matcher slice

Pair-level analysis reproduces the article headline; document-level analysis is a sensitivity check on the 6 source documents.

,n,mean_control,mean_feedback,absolute_delta,relative_delta_pct,p_value,relative_ci_95_pct
analysis,,,,,,,
pair-level headline effect,505,0.4172,0.4886,0.0715,17.13,0.00010,[8.1%; 26.7%]
document-level sensitivity check,6,0.4196,0.4758,0.0562,13.39,0.53147,[-16.4%; 62.2%]



## Supporting intrinsic schema quality

Это не основной тезис статьи, но полезный supporting result.

Ниже считается простая intrinsic schema-quality эвристика по уже извлечённым табличным элементам.
Она не использует alignment, а просто показывает, что `feedback` не только повышает часть alignment-centric метрик,
но и делает сами схемы более явными и менее generic по naming.


In [9]:
semantic_quality_rows = []
for model_id, frame in elements.groupby('model_id'):
    info = model_index.loc[model_index['model_id'] == model_id].iloc[0]
    named = frame[frame['element_kind'].isin(['type', 'property', 'relation'])].copy()
    identifier_specificity = (
        float(named['name'].fillna('').astype(str).map(tokenize_identifier).map(len).mean())
        if len(named)
        else 0.0
    )
    structure_explicitness = float(
        frame['symbol_path']
        .fillna('')
        .astype(str)
        .map(lambda value: len([part for part in value.split('.') if part]))
        .mean()
    )
    bool_properties = frame[
        (frame['element_kind'] == 'property')
        & (frame['normalized_type'].fillna('').astype(str) == 'bool')
    ].copy()
    bool_naming_ok_rate = (
        float(
            bool_properties['name']
            .fillna('')
            .astype(str)
            .map(
                lambda name: any(
                    name.lower().startswith(prefix)
                    for prefix in BOOLEAN_PREFIXES
                )
            )
            .mean()
        )
        if len(bool_properties)
        else 1.0
    )
    properties = frame[frame['element_kind'] == 'property'].copy()
    properties['role_key'] = (
        properties['parent_symbol_path'].fillna('').astype(str)
        + '::'
        + properties['name'].fillna('').astype(str).str.lower()
    )
    duplicate_name_ok_rate = 1.0 - (
        float(properties['role_key'].duplicated().sum()) / max(1, len(properties))
    )
    generic_name_rate = (
        float(
            named['name']
            .fillna('')
            .astype(str)
            .map(lambda name: name.lower() in GENERIC_NAMES)
            .mean()
        )
        if len(named)
        else 0.0
    )
    semantic_quality_rows.append(
        {
            'model_id': model_id,
            'source_document_id': info['source_document_id'],
            'condition': info['condition'],
            'producer_id': info['producer_id'],
            'identifier_specificity': identifier_specificity,
            'structure_explicitness': structure_explicitness,
            'bool_naming_ok_rate': bool_naming_ok_rate,
            'duplicate_name_ok_rate': duplicate_name_ok_rate,
            'generic_name_rate': generic_name_rate,
        }
    )

semantic_quality = pd.DataFrame(semantic_quality_rows)
positive_columns = [
    'identifier_specificity',
    'structure_explicitness',
    'bool_naming_ok_rate',
    'duplicate_name_ok_rate',
]
for column in positive_columns:
    semantic_quality[f'{column}_rank'] = semantic_quality[column].rank(
        pct=True,
        method='average',
    )
semantic_quality['generic_name_rate_rank'] = 1.0 - semantic_quality[
    'generic_name_rate'
].rank(pct=True, method='average')
semantic_quality['semantic_quality_index'] = semantic_quality[
    [f'{column}_rank' for column in positive_columns] + ['generic_name_rate_rank']
].mean(axis=1)

semantic_quality_overall = (
    semantic_quality.groupby('condition')['semantic_quality_index']
    .agg(['mean', 'median', 'min', 'max'])
    .reindex(SCENARIO_ORDER)
    .round(4)
)
semantic_quality_by_document = (
    semantic_quality.groupby(['source_document_id', 'condition'])['semantic_quality_index']
    .mean()
    .reset_index()
    .pivot_table(
        index='source_document_id',
        columns='condition',
        values='semantic_quality_index',
    )
    .reindex(columns=SCENARIO_ORDER)
    .round(4)
)

display(Markdown('### Supporting semantic quality index (overall)'))
display(semantic_quality_overall)
display(Markdown('### Supporting semantic quality index by source document'))
display(semantic_quality_by_document)

### Supporting semantic quality index (overall)

,mean,median,min,max
condition,,,,
control,0.4532,0.4435,0.3630,0.5667
guided,0.5070,0.5324,0.3130,0.6093
feedback,0.5565,0.5852,0.4167,0.6333


### Supporting semantic quality index by source document

condition,control,guided,feedback
source_document_id,,,
framework-laptop-13-specs,0.5204,0.5543,0.5821
github-create-issue-api,0.4568,0.4944,0.5549
ikea-billy-mainual,0.4679,0.5049,0.5722
king-arthur-pancakes,0.3821,0.4778,0.5938
standard-fragment,0.4673,0.5858,0.5809
stripe-payment-intent,0.4247,0.4247,0.4549



## Summary for the article

Эта тетрадь фиксирует три ключевых факта:

1. **Общая same-scenario structural coverage** растёт монотонно: `control < guided < feedback` по `coverage_f1`.
2. Если ограничиться более semantic/structure-aware matcher'ами, то и **same-scenario Pair alignment F1** становится монотонным: `control < guided < feedback`.
3. Самый наглядный alignment table в текущем эксперименте — это `valentine:coma_py` по directed model pairs, где **все 6 pair comparisons** дают `control < guided < feedback`.

При этом эксперимент остаётся честным:
- не все документы выигрывают от `feedback` одинаково сильно;
- не все alignment methods поддерживают гипотезу одинаково;
- часть lexical methods штрафует `feedback` за более активное переименование сущностей.
